# Data-Leakage Sanity Check — Negative Controls
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

## Why this notebook exists

Every model in this project scores **above 95 %** F1 / accuracy. That is suspiciously high,
and high scores are exactly where **data leakage** hides. This notebook runs **negative
controls** to prove the pipeline itself is not cheating.

## The logic (a placebo test)

If we train a model on data that contains **no real feature→label relationship**, a *correct*
pipeline must score at **chance level**:

- binary (2 classes) → F1 / accuracy ≈ **0.50**
- multi-class (7 classes) → macro-F1 ≈ **1/7 ≈ 0.14**

If it scores meaningfully *above* chance on signal-free data, something is leaking — e.g. test
labels bleeding into training, evaluation on training data, or a transform fit on the whole
dataset.

We run three conditions and compare them side by side:

| Condition | Features | Labels | Expected score |
|---|---|---|---|
| **A — REAL (control)** | real | real | high (~0.99) — our actual result |
| **B — label shuffle** | real | **shuffled** | chance — destroys the feature→label link |
| **C — random synthetic** | **random, matched mean/std** | random | chance — no signal at all |

**A passes if it's high; B and C pass if they're at chance.** All three passing = the
pipeline is sound and the 99 % is real signal, not a harness bug.

## What this test does and does NOT catch

- **Catches:** harness bugs — test-label leakage, train/test contamination, evaluating on
  training data, a scaler/encoder fit on the full dataset before splitting.
- **Does NOT catch:** a feature that is *legitimately but unrealistically* informative
  (e.g. `Destination Port` memorising which service each attack targeted). Shuffling labels
  destroys that relationship too, so the control still passes. That class of problem needs a
  separate feature-ablation test.

**Kaggle setup:** attach the FeatureSelection output dataset; set `IN_DIR`. No GPU needed.

## 1. Imports & Config

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CLASS_NAMES = ['BENIGN', 'Bot/Infiltration', 'Brute Force', 'DDoS', 'DoS', 'PortScan', 'Web Attack']
N_CLASSES   = len(CLASS_NAMES)

IN_DIR      = '/kaggle/input/ids-selected'   # EDIT to your FeatureSelection mount path
OUT_DIR     = '/kaggle/working'
FIGURES_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

_report_lines = []
def _log(t=''):
    _report_lines.append(str(t)); print(t)
def _savefig(name, fig=None):
    p = os.path.join(FIGURES_DIR, name); (fig or plt).savefig(p, dpi=130, bbox_inches='tight'); return p
def write_report():
    p = os.path.join(OUT_DIR, 'Leakage_SanityCheck_Report.txt')
    with open(p, 'w', encoding='utf-8') as f: f.write('\n'.join(_report_lines))
    print(f'\nReport saved -> {p}')

_log('=' * 70)
_log('DATA-LEAKAGE SANITY CHECK (NEGATIVE CONTROLS)  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
_log('=' * 70)
print('Setup complete.')

## 2. Load Data & Define a Fast, Honest Evaluator
We use a **fast model** (lightweight Random Forest / logistic regression) and a **subsample**
for these controls — we are not chasing peak accuracy, we just need a model strong enough
that, *if there were leakage, it would exploit it*. We deliberately re-split inside each
condition so the control is fully self-contained and cannot accidentally reuse a leaked
split.

In [ ]:
train_path = os.path.join(IN_DIR, 'train_selected.parquet')
feat_path  = os.path.join(IN_DIR, 'selected_features.json')
for p in [train_path, feat_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f'{p} not found. Set IN_DIR to your FeatureSelection mount path.')

with open(feat_path) as f:
    selected_features = json.load(f)['selected_features']

df = pd.read_parquet(train_path)
n_features = len(selected_features)

# subsample for speed — stratified so all 7 classes are present
SAMPLE_N = 200_000
samp = (df.groupby('label_multi', group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max(1, SAMPLE_N * len(g) // len(df))),
                                    random_state=RANDOM_SEED)))
X_all      = samp[selected_features].values.astype(np.float32)
y_all_bin  = samp['label_binary'].values.astype(np.int64)
y_all_mult = samp['label_multi'].values.astype(np.int64)

_log('')
_log('── SECTION 2 : DATA LOADED ────────────────────────────────')
_log(f'  Features            : {n_features}')
_log(f'  Subsample for tests : {len(X_all):,} rows (stratified)')
_log(f'  Chance level binary : {1/2:.4f}  (F1/acc of a no-signal model)')
_log(f'  Chance level 7-class: {1/N_CLASSES:.4f}  (macro-F1 of a no-signal model)')

def quick_eval(X, y, task, label=''):
    """Fresh stratified split + fast RF; returns (f1, acc). Self-contained per call."""
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25,
                                          random_state=RANDOM_SEED, stratify=y)
    clf = RandomForestClassifier(n_estimators=80, max_depth=None, n_jobs=-1,
                                 class_weight='balanced', random_state=RANDOM_SEED)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    if task == 'binary':
        f1 = f1_score(yte, pred); acc = accuracy_score(yte, pred)
    else:
        f1 = f1_score(yte, pred, average='macro'); acc = accuracy_score(yte, pred)
    return f1, acc

## 3. Condition A — REAL data (the control we *want* to be high)
Train on the genuine features and genuine labels. This should reproduce the high scores we've
seen throughout the project — confirming the model genuinely works when real signal is present.

In [ ]:
_log('')
_log('── SECTION 3 : CONDITION A — REAL DATA ────────────────────')
t0 = time.time()
realA_bin_f1,  realA_bin_acc  = quick_eval(X_all, y_all_bin,  'binary')
realA_mult_f1, realA_mult_acc = quick_eval(X_all, y_all_mult, 'multi')
_log(f'  Binary      : F1={realA_bin_f1:.4f}   acc={realA_bin_acc:.4f}')
_log(f'  Multi-class : macroF1={realA_mult_f1:.4f}   acc={realA_mult_acc:.4f}')
_log(f'  (computed in {time.time()-t0:.1f}s — expected HIGH; this is the positive control)')

## 4. Condition B — LABEL SHUFFLE (real features, scrambled labels)
Keep the real feature matrix exactly as-is, but randomly permute the labels. This **destroys
the feature→label relationship** while keeping the real feature distribution intact. A correct
pipeline can no longer do better than chance, because there is nothing left to learn.

We repeat the shuffle over a few random seeds and report mean ± spread, so a single lucky
shuffle can't mislead us.

In [ ]:
_log('')
_log('── SECTION 4 : CONDITION B — LABEL SHUFFLE ────────────────')
shuf_bin_f1s, shuf_mult_f1s = [], []
shuf_bin_accs, shuf_mult_accs = [], []
for seed in [1, 2, 3]:
    rng = np.random.RandomState(seed)
    yb_shuf = y_all_bin.copy();  rng.shuffle(yb_shuf)
    ym_shuf = y_all_mult.copy(); rng.shuffle(ym_shuf)
    f1b, accb = quick_eval(X_all, yb_shuf, 'binary')
    f1m, accm = quick_eval(X_all, ym_shuf, 'multi')
    shuf_bin_f1s.append(f1b);  shuf_bin_accs.append(accb)
    shuf_mult_f1s.append(f1m); shuf_mult_accs.append(accm)
    _log(f'  seed {seed}: binary F1={f1b:.4f} acc={accb:.4f} | multi macroF1={f1m:.4f} acc={accm:.4f}')

shufB_bin_f1   = np.mean(shuf_bin_f1s)
shufB_mult_f1  = np.mean(shuf_mult_f1s)
shufB_bin_acc  = np.mean(shuf_bin_accs)
shufB_mult_acc = np.mean(shuf_mult_accs)
_log('')
_log(f'  MEAN binary      : F1={shufB_bin_f1:.4f}   acc={shufB_bin_acc:.4f}   (chance acc ~ majority class freq)')
_log(f'  MEAN multi-class : macroF1={shufB_mult_f1:.4f}   acc={shufB_mult_acc:.4f}   (chance macroF1 ~ {1/N_CLASSES:.3f})')

## 5. Condition C — RANDOM SYNTHETIC data
Generate an **entirely artificial dataset** with the same shape and the same per-column mean /
standard deviation as the real (scaled) features — but every value is drawn independently from
a normal distribution, and labels are assigned at random in the real class proportions. There
is *no* relationship between any feature and the label by construction.

This is the strongest negative control: even the feature *correlations* are gone (each column
is independent Gaussian noise). A model can only score at chance.

In [ ]:
_log('')
_log('── SECTION 5 : CONDITION C — RANDOM SYNTHETIC DATA ────────')
rng = np.random.RandomState(RANDOM_SEED)
n_syn = len(X_all)

# match the per-column mean and std of the real features
col_mean = X_all.mean(axis=0)
col_std  = X_all.std(axis=0) + 1e-9
X_syn = (rng.randn(n_syn, n_features).astype(np.float32) * col_std + col_mean)

# random labels in the real class proportions
bin_p  = np.bincount(y_all_bin)  / len(y_all_bin)
mult_p = np.bincount(y_all_mult, minlength=N_CLASSES) / len(y_all_mult)
y_syn_bin  = rng.choice(len(bin_p),  size=n_syn, p=bin_p).astype(np.int64)
y_syn_mult = rng.choice(N_CLASSES,   size=n_syn, p=mult_p).astype(np.int64)

synC_bin_f1,  synC_bin_acc  = quick_eval(X_syn, y_syn_bin,  'binary')
synC_mult_f1, synC_mult_acc = quick_eval(X_syn, y_syn_mult, 'multi')
_log(f'  Synthetic shape : {X_syn.shape}  (mean/std matched to real, values = pure noise)')
_log(f'  Binary      : F1={synC_bin_f1:.4f}   acc={synC_bin_acc:.4f}')
_log(f'  Multi-class : macroF1={synC_mult_f1:.4f}   acc={synC_mult_acc:.4f}')
_log(f'  (expected at chance — no feature carries any label information)')

## 6. Verdict — Side-by-Side
The whole test in one place. A clean pipeline shows: **A high, B and C at chance.**

In [ ]:
summary = pd.DataFrame({
    'Binary F1':       [realA_bin_f1,  shufB_bin_f1,  synC_bin_f1],
    'Binary acc':      [realA_bin_acc, shufB_bin_acc, synC_bin_acc],
    'Multi macro-F1':  [realA_mult_f1, shufB_mult_f1, synC_mult_f1],
    'Multi acc':       [realA_mult_acc,shufB_mult_acc,synC_mult_acc],
}, index=['A: REAL', 'B: label-shuffle', 'C: random-synthetic'])

_log('')
_log('=' * 70)
_log('SECTION 6 : VERDICT')
_log('=' * 70)
_log(summary.to_string())
display(summary)

# automated pass/fail
chance_bin  = max(bin_p)            # a no-signal model's best accuracy = majority-class freq
chance_mult = 1.0 / N_CLASSES
TOL = 0.10                          # tolerance above chance before we flag a problem

controls_low = (
    shufB_mult_f1 < chance_mult + TOL and synC_mult_f1 < chance_mult + TOL and
    shufB_bin_f1  < 0.5 + TOL        and synC_bin_f1  < 0.5 + TOL
)
real_high = realA_mult_f1 > 0.80 and realA_bin_f1 > 0.90

_log('')
if real_high and controls_low:
    verdict = 'PASS — pipeline is sound. Real data scores high; both no-signal controls collapse to chance.'
elif not controls_low:
    verdict = 'FAIL — a no-signal control scored well above chance. Investigate leakage in the harness.'
else:
    verdict = 'INCONCLUSIVE — real data did not score as high as expected; re-check the setup.'
_log('VERDICT: ' + verdict)
print('\n' + verdict)

# plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
conds = ['A: REAL', 'B: shuffle', 'C: synthetic']
colors = ['#2E7D32', '#EF6C00', '#C62828']

ax = axes[0]
ax.bar(conds, [realA_bin_f1, shufB_bin_f1, synC_bin_f1], color=colors)
ax.axhline(0.5, ls='--', color='gray', label='chance F1 = 0.50')
ax.set_title('Binary F1 — real vs no-signal controls', fontweight='bold')
ax.set_ylim(0, 1.05); ax.legend()
for i, v in enumerate([realA_bin_f1, shufB_bin_f1, synC_bin_f1]):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

ax = axes[1]
ax.bar(conds, [realA_mult_f1, shufB_mult_f1, synC_mult_f1], color=colors)
ax.axhline(chance_mult, ls='--', color='gray', label=f'chance macro-F1 = {chance_mult:.3f}')
ax.set_title('Multi-class macro-F1 — real vs no-signal controls', fontweight='bold')
ax.set_ylim(0, 1.05); ax.legend()
for i, v in enumerate([realA_mult_f1, shufB_mult_f1, synC_mult_f1]):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.suptitle('Leakage Negative-Control Test  —  controls must collapse to chance',
             fontsize=14, fontweight='bold')
plt.tight_layout(); _savefig('01_leakage_controls.png', fig); plt.show()

summary.to_csv(os.path.join(OUT_DIR, 'leakage_sanity_summary.csv'))
write_report()
print(f'\nSaved -> {OUT_DIR}/leakage_sanity_summary.csv')